# Preprocessing & Feature Engineering

## Objective
Prepare data for time series forecasting with:
1. Chronological train/test split (no data leakage)
2. Lag features (past values)
3. Rolling window statistics
4. Calendar features
5. Promotion features

## Critical Principles
- **No data leakage**: Future information never used in past predictions
- **Chronological split**: Train/test split respects time ordering
- **Proper lag/rolling**: All features use only historical values
- **Continuous features**: Features created across train-test boundary correctly

In [1]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import project modules
import src.config as config
from src.utils import print_section_header, describe_dataframe
from src.preprocessing import (
    split_time_series,
    build_features,
    handle_missing_from_features,
    prepare_model_data,
    get_feature_names,
    create_train_test_features
)

# Set random seed
np.random.seed(config.RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')

print("Environment setup complete!")

Environment setup complete!


## 1. Load Processed Data from EDA

In [2]:
# Load the processed daily sales data from EDA notebook
data_path = config.DATA_PATH / 'processed_daily_sales.csv'
daily_sales = pd.read_csv(data_path, parse_dates=['date'])

print(f"Data loaded: {daily_sales.shape}")
print(f"Date range: {daily_sales['date'].min()} to {daily_sales['date'].max()}")
print(f"\nFirst few rows:")
daily_sales.head()

Data loaded: (1688, 3)
Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00

First few rows:


,date,sales,onpromotion
0,2013-01-01,0.0,0
1,2013-01-02,10686.0,0
2,2013-01-03,7342.0,0
3,2013-01-04,7250.0,0
4,2013-01-05,10699.0,0


In [3]:
# Check data summary
describe_dataframe(daily_sales, name="Daily Sales Data")


 Daily Sales Data Summary

Shape: 1688 rows × 3 columns

Column types:
date           datetime64[us]
sales                 float64
onpromotion             int64
dtype: object

Memory usage: 0.04 MB

Missing values:
Empty DataFrame
Columns: [Missing, Percentage]
Index: []
No missing values found.


## 2. Chronological Train/Test Split

**Critical**: We use chronological split (NOT random shuffling) to:
- Respect temporal ordering
- Prevent data leakage
- Simulate real-world forecasting scenario

The test set represents the future that we want to predict.

In [4]:
# Perform chronological split
train_df, test_df, split_date = split_time_series(
    daily_sales,
    date_col='date',
    test_ratio=config.TEST_RATIO,
    verbose=True
)


 Time Series Split

Total rows: 1688
Train rows: 1434 (85.0%)
Test rows: 254 (15.0%)

Train date range: 2013-01-01 00:00:00 to 2016-12-04 00:00:00
Test date range: 2016-12-05 00:00:00 to 2017-08-15 00:00:00

Split date: 2016-12-04 00:00:00

✓ Chronological split - no data leakage


In [ ]:
# Visualize the split
fig, ax = plt.subplots(figsize=(14, 6))

# Plot train and test
ax.plot(train_df['date'], train_df['sales'], label='Train', alpha=0.7, linewidth=1)
ax.plot(test_df['date'], test_df['sales'], label='Test', alpha=0.7, linewidth=1, color='orange')

# Mark split point
ax.axvline(split_date, color='red', linestyle='--', linewidth=2, label='Split Point')

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Sales', fontsize=12)
ax.set_title('Train/Test Split (Chronological)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()

# Save figure
from src.utils import save_figure
save_figure(fig, '11_train_test_split.png')
plt.show()

## 3. Feature Engineering

### Feature Types:

1. **Lag Features**: Sales from previous days (1, 7, 14, 30 days ago)
   - Captures recent history
   - `sales_lag_1`, `sales_lag_7`, etc.

2. **Rolling Window Features**: Moving averages and standard deviations
   - Smooths out noise
   - Captures trends
   - `rolling_mean_7`, `rolling_std_14`, etc.

3. **Calendar Features**: Day of week, month, year, weekend indicator
   - Captures seasonality
   - `day_of_week`, `month`, `is_weekend`, etc.

4. **Promotion Features**: Current and historical promotion information
   - `onpromotion`, `has_promotion`, `rolling_promo_7`

### Leakage Prevention:
- All lag features use `.shift()` correctly
- Rolling features exclude current day (shift before rolling)
- Test set features can use train set history (correct behavior)

In [6]:
# Create features for both train and test
# This function handles the train-test boundary correctly
train_features, test_features, feature_names = create_train_test_features(
    train_df,
    test_df,
    target_col='sales',
    date_col='date',
    lags=config.LAG_FEATURES,
    roll_windows=config.ROLLING_WINDOWS,
    verbose=True
)


 Creating Train/Test Features

Train size: 1434
Test size: 254

 Feature Engineering

Creating lag features...
Creating rolling window features...
Creating calendar features...
Creating promotion features...

✓ Feature engineering complete
Original columns: 4
Total columns after engineering: 24
New features created: 20

 Handling Missing Values

Rows before: 1688

Missing values per column:
sales_lag_1         1
sales_lag_7         7
sales_lag_14       14
sales_lag_30       30
rolling_mean_7      7
rolling_std_7       7
rolling_mean_14    14
rolling_std_14     14
rolling_mean_30    30
rolling_std_30     30
rolling_promo_7     1
dtype: int64

✓ Dropped rows with missing values
Rows after: 1658
Rows dropped: 30

✓ Train features shape: (1404, 23)
✓ Test features shape: (254, 23)
✓ Number of features: 21


In [7]:
# Inspect feature names
print_section_header("Feature List")
print(f"Total features: {len(feature_names)}\n")
for i, feat in enumerate(feature_names, 1):
    print(f"{i:2d}. {feat}")


 Feature List

Total features: 21

 1. sales_lag_1
 2. sales_lag_7
 3. sales_lag_14
 4. sales_lag_30
 5. rolling_mean_7
 6. rolling_std_7
 7. rolling_mean_14
 8. rolling_std_14
 9. rolling_mean_30
10. rolling_std_30
11. day_of_week
12. day_of_month
13. month
14. year
15. quarter
16. is_weekend
17. is_month_start
18. is_month_end
19. onpromotion
20. has_promotion
21. rolling_promo_7


In [8]:
# Preview train features
print_section_header("Training Features Preview")
print(train_features.head(40))  # Show first 40 rows to see how features populate


 Training Features Preview

         date    sales  onpromotion  sales_lag_1  sales_lag_7  sales_lag_14  \
0  2013-01-31   4268.0            0       5669.0       4007.0        4998.0   
1  2013-02-01   7265.0            0       4268.0       5895.0        6000.0   
2  2013-02-02  11068.0            0       7265.0       8709.0        9628.0   
3  2013-02-03  11125.0            0      11068.0       9667.0       10488.0   
4  2013-02-04   6202.0            0      11125.0       5471.0        5932.0   
5  2013-02-05   5903.0            0       6202.0       4745.0       19263.0   
6  2013-02-06   7160.0            0       5903.0       5669.0        5869.0   
7  2013-02-07   5444.0            0       7160.0       4268.0        4007.0   
8  2013-02-08   5826.0            0       5444.0       7265.0        5895.0   
9  2013-02-09   7683.0            0       5826.0      11068.0        8709.0   
10 2013-02-10   5655.0            0       7683.0      11125.0        9667.0   
11 2013-02-11   5790.0 

In [9]:
# Preview test features
print_section_header("Test Features Preview")
print(test_features.head(10))


 Test Features Preview

           date    sales  onpromotion  sales_lag_1  sales_lag_7  sales_lag_14  \
1404 2016-12-05  11826.0          114      19720.0       9688.0        9749.0   
1405 2016-12-06  16153.0          122      11826.0       9257.0        8482.0   
1406 2016-12-07  12334.0          202      16153.0      11083.0       10780.0   
1407 2016-12-08  11796.0          137      12334.0      10704.0        8042.0   
1408 2016-12-09  12602.0          149      11796.0      13883.0       11198.0   
1409 2016-12-10  14516.0          148      12602.0      15763.0       14333.0   
1410 2016-12-11  17866.0          146      14516.0      19720.0       17845.0   
1411 2016-12-12  15947.0          120      17866.0      11826.0        9688.0   
1412 2016-12-13  12429.0          150      15947.0      16153.0        9257.0   
1413 2016-12-14  17827.0          138      12429.0      12334.0       11083.0   

      sales_lag_30  rolling_mean_7  rolling_std_7  rolling_mean_14  ...  \
1404    

## 4. Prepare Model Matrices

Extract X (features) and y (target) for model training and evaluation.

In [10]:
# Prepare train data
X_train, y_train, train_dates = prepare_model_data(
    train_features,
    feature_cols=feature_names,
    target_col='sales',
    date_col='date',
    return_dates=True
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"train_dates shape: {train_dates.shape}")

X_train shape: (1404, 21)
y_train shape: (1404,)
train_dates shape: (1404,)


In [11]:
# Prepare test data
X_test, y_test, test_dates = prepare_model_data(
    test_features,
    feature_cols=feature_names,
    target_col='sales',
    date_col='date',
    return_dates=True
)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"test_dates shape: {test_dates.shape}")

X_test shape: (254, 21)
y_test shape: (254,)
test_dates shape: (254,)


In [12]:
# Display feature statistics
print_section_header("Feature Statistics (Training Set)")
feature_stats = pd.DataFrame(X_train, columns=feature_names).describe().T
feature_stats


 Feature Statistics (Training Set)



,count,mean,std,min,25%,50%,75%,max
sales_lag_1,1404.0,9463.175953,3518.919813,0.000000,7117.500000,8554.000000,11439.250000,46271.000000
sales_lag_7,1404.0,9440.466551,3520.651951,0.000000,7100.750000,8532.500000,11412.000000,46271.000000
sales_lag_14,1404.0,9427.467264,3525.870151,0.000000,7089.250000,8527.500000,11390.250000,46271.000000
sales_lag_30,1404.0,9373.993617,3509.574978,0.000000,7070.750000,8486.500000,11334.750000,46271.000000
rolling_mean_7,1404.0,9449.760914,1720.563809,6208.714286,8433.678571,9291.214286,9993.678571,21072.571429
rolling_std_7,1404.0,3009.600246,1385.297794,835.222297,2326.013384,2758.582157,3282.966732,14411.384617
rolling_mean_14,1404.0,9440.841806,1471.926090,6677.428571,8502.821429,9198.642857,9979.803571,16182.785714
rolling_std_14,1404.0,3081.004984,1229.171177,1460.642507,2479.067521,2824.837555,3224.093946,10667.569240
rolling_mean_30,1404.0,9418.139533,1251.693673,7197.333333,8587.033333,9277.866667,10160.300000,13539.500000
rolling_std_30,1404.0,3157.725761,1091.079068,1563.561962,2564.174690,2834.719141,3428.398808,8002.121303


## 5. Feature Correlation Analysis

In [13]:
# Create correlation matrix with target
train_with_target = pd.DataFrame(X_train, columns=feature_names)
train_with_target['sales'] = y_train

# Compute correlation with target
correlations = train_with_target.corr()['sales'].drop('sales').sort_values(ascending=False)

print_section_header("Feature Correlation with Target (Sales)")
print(correlations)


 Feature Correlation with Target (Sales)

is_weekend         0.567690
sales_lag_7        0.483330
sales_lag_14       0.441869
day_of_week        0.412059
sales_lag_1        0.350764
rolling_mean_7     0.310723
rolling_mean_14    0.286288
rolling_mean_30    0.259058
has_promotion      0.217261
onpromotion        0.191754
month              0.187625
year               0.187298
rolling_promo_7    0.186554
quarter            0.179277
rolling_std_7      0.119902
rolling_std_30     0.096333
rolling_std_14     0.088889
is_month_start     0.061050
is_month_end      -0.001219
sales_lag_30      -0.076989
day_of_month      -0.135118
Name: sales, dtype: float64


In [ ]:
# Visualize top correlations
fig, ax = plt.subplots(figsize=(10, 8))

correlations.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Correlation with Sales', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title('Feature Correlation with Target', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.axvline(0, color='black', linewidth=0.8)

plt.tight_layout()
save_figure(fig, '12_feature_correlations.png')
plt.show()

## 6. Save Processed Data for Modeling

In [15]:
# Save processed datasets
print_section_header("Saving Processed Data")

# Save complete feature dataframes
train_features.to_csv(config.DATA_PATH / 'train_features.csv', index=False)
test_features.to_csv(config.DATA_PATH / 'test_features.csv', index=False)

# Save numpy arrays for quick loading
np.save(config.DATA_PATH / 'X_train.npy', X_train)
np.save(config.DATA_PATH / 'y_train.npy', y_train)
np.save(config.DATA_PATH / 'X_test.npy', X_test)
np.save(config.DATA_PATH / 'y_test.npy', y_test)
np.save(config.DATA_PATH / 'train_dates.npy', train_dates)
np.save(config.DATA_PATH / 'test_dates.npy', test_dates)

# Save feature names
with open(config.DATA_PATH / 'feature_names.txt', 'w') as f:
    for feat in feature_names:
        f.write(f"{feat}\n")

print("✓ Saved:")
print("  - train_features.csv")
print("  - test_features.csv")
print("  - X_train.npy, y_train.npy")
print("  - X_test.npy, y_test.npy")
print("  - train_dates.npy, test_dates.npy")
print("  - feature_names.txt")


 Saving Processed Data

✓ Saved:
  - train_features.csv
  - test_features.csv
  - X_train.npy, y_train.npy
  - X_test.npy, y_test.npy
  - train_dates.npy, test_dates.npy
  - feature_names.txt


## 7. Summary

### Data Preparation Complete

**Train Set:**
- Rows: {}
- Features: {}
- Date range: {} to {}

**Test Set:**
- Rows: {}
- Features: {}
- Date range: {} to {}

### Feature Categories:
1. **Lag features (4)**: 1, 7, 14, 30 days
2. **Rolling mean (3)**: 7, 14, 30 day windows
3. **Rolling std (3)**: 7, 14, 30 day windows
4. **Calendar features (8)**: day of week, month, year, etc.
5. **Promotion features (3)**: onpromotion, has_promotion, rolling_promo_7

### Data Quality Checks:
✓ No missing values after feature engineering
✓ Chronological split maintained
✓ No data leakage in feature creation
✓ Features properly aligned with target

### Next Steps:
1. Build baseline models (Naive, Seasonal Naive, Moving Average)
2. Train statistical models (SARIMA/SARIMAX)
3. Train ML models (RandomForest, GradientBoosting)
4. Compare and evaluate all models

In [16]:
# Print final summary with actual numbers
print_section_header("Final Summary")
print(f"Train Set: {X_train.shape[0]} rows × {X_train.shape[1]} features")
print(f"  Date range: {pd.to_datetime(train_dates[0])} to {pd.to_datetime(train_dates[-1])}")
print(f"\nTest Set: {X_test.shape[0]} rows × {X_test.shape[1]} features")
print(f"  Date range: {pd.to_datetime(test_dates[0])} to {pd.to_datetime(test_dates[-1])}")
print(f"\nTotal features: {len(feature_names)}")
print(f"\n✓ Ready for modeling!")


 Final Summary

Train Set: 1404 rows × 21 features
  Date range: 2013-01-31 00:00:00 to 2016-12-04 00:00:00

Test Set: 254 rows × 21 features
  Date range: 2016-12-05 00:00:00 to 2017-08-15 00:00:00

Total features: 21

✓ Ready for modeling!
